In [96]:
import duckdb
import pandas as pd

con = duckdb.connect()

# Foul Study  
**Currently the 7s/11s shot quality buckets do not accurately reflect (enough) the value of what occurs. An effort to correct this is in the investigation below.**

The file below contains additional data tracked after the season ended with a 'Type' column for both Home and Away collected by rewatching all 141 games from the Mountain East Conference season.  
In the 'Type' column(s):
- Fouls are tracked via:
    - F2: Foul on 2PA (not an And-1)
    - F2A: Foul on an And-1 2PA
    - F3: Foul on 3PA (not an And-1)
    - F3A: Foul on an And-1 3PA
    - OAO: One-and-One Bonus
    - DB: Double Bonus
    - T1: One-Shot Technical
    - T2: Two-Shot Technical

In [97]:
file = "Fouls_and_TOs.csv"

In [98]:
data_q = f"""
    SELECT Date, Home_Team, Home_Type, Home_Points, Away_Team, Away_Type, Away_Points FROM {file}
    WHERE Date IS NOT NULL
"""
data = con.query(data_q)

### Brief Investigation into Foul Types Drawn by Team ###

In [99]:
foul_h_query = f"""
    SELECT Home_Team, Home_Type, sum(Home_Points) as H_Total_Points, count(Home_Points) as H_Total_Shots FROM data
    WHERE Home_Type in ('T1', 'T2', 'F2', 'F2A', 'F3', 'F3A', 'OAO', 'DB')
    GROUP BY Home_Team, Home_Type
"""

con.sql(foul_h_query).show()
foul_a_query = f"""
    SELECT Away_Team, Away_Type, sum(Away_Points) as 'A_Total_Points', count(Away_Points) as 'A_Total_Shots' FROM data
    WHERE Away_Type in ('T1', 'T2', 'F2', 'F2A', 'F3', 'F3A', 'OAO', 'DB')
    GROUP BY Away_Team, Away_Type
"""

con.sql(foul_a_query)
team_totals = f"""
    WITH Home as ({foul_h_query}),
    Away as ({foul_a_query})
    SELECT COALESCE(h.Home_Team, a.Away_Team) as Team, 
        COALESCE (h.Home_Type, a.Away_Type) as Type, 
        IFNULL(h.H_Total_Points, 0) Home_Points, 
        IFNULL(H_Total_Shots, 0) Home_Shots, 
        IFNULL(a.A_Total_Points, 0) Away_Points, 
        IFNULL(a.A_Total_Shots, 0) Away_Shots,
        (Home_Points+Away_Points) Total_Points,
        (Home_Shots+Away_Shots) Total_Shots
    FROM Home h
    FULL JOIN Away a
    ON h.Home_Team = a.Away_Team
    AND h.Home_Type = a.Away_Type
    ORDER BY Team, Type
"""

con.sql(team_totals)

┌──────────────┬───────────┬────────────────┬───────────────┐
│  Home_Team   │ Home_Type │ H_Total_Points │ H_Total_Shots │
│   varchar    │  varchar  │     int128     │     int64     │
├──────────────┼───────────┼────────────────┼───────────────┤
│ Fairmont     │ F2        │            117 │            87 │
│ Fairmont     │ F2A       │             39 │            15 │
│ Fairmont     │ F3A       │             12 │             3 │
│ Fairmont     │ OAO       │             23 │            19 │
│ Charleston   │ F2A       │             50 │            18 │
│ Charleston   │ F2        │             80 │            56 │
│ Charleston   │ OAO       │             30 │            22 │
│ Charleston   │ DB        │             41 │            24 │
│ WVSU         │ F2        │            104 │            71 │
│ WVSU         │ OAO       │             31 │            21 │
│  ·           │ ·         │              · │             · │
│  ·           │ ·         │              · │             · │
│  ·    

┌──────────────┬─────────┬─────────────┬────────────┬─────────────┬────────────┬──────────────┬─────────────┐
│     Team     │  Type   │ Home_Points │ Home_Shots │ Away_Points │ Away_Shots │ Total_Points │ Total_Shots │
│   varchar    │ varchar │   int128    │   int64    │   int128    │   int64    │    int128    │    int64    │
├──────────────┼─────────┼─────────────┼────────────┼─────────────┼────────────┼──────────────┼─────────────┤
│ Charleston   │ DB      │          41 │         24 │          15 │         15 │           56 │          39 │
│ Charleston   │ F2      │          80 │         56 │          83 │         60 │          163 │         116 │
│ Charleston   │ F2A     │          50 │         18 │          36 │         13 │           86 │          31 │
│ Charleston   │ F3      │           7 │          3 │           0 │          0 │            7 │           3 │
│ Charleston   │ OAO     │          30 │         22 │          26 │         21 │           56 │          43 │
│ Charlest

### Proactively implementing a t-test function for future use in the investigative study ###

In [100]:
import numpy as np
from scipy import stats as sps

# Register a UDF DuckDB can call directly in SQL
def t_pvalue(t_stat: float, df: float) -> float:
    if df is None or df <= 0 or t_stat is None:
        return None
    return float(2 * sps.t.sf(abs(t_stat), df))  # two-tailed

con.create_function('t_pvalue', t_pvalue, ['DOUBLE', 'DOUBLE'], 'DOUBLE')

## This code creates an output of each foul buckets expectations versus a "mathematical expectation" ##
For each bucket the expecation can be derived as follows based on league-wide 75% FT%:
- One Shot Tech (T1): (1 FT)(75%) = 0.75 PPS
- Two Shot Tech (T2): (2 FT)(75%) = 1.50 PPS
- Foul on non-And-1 2PA (2P): (2 FT)(75%) = 1.50 PPS
- Foul on And-1 2PA (2PA): 2 + (1 FT)(75%) = 2.75 PPS
- Foul on non-And-1 3PA (3P): (3 FT)(75%) = 2.25 PPS
- Foul on And-1 3PA (3PA): 3 + (1 FT)(75%) = 3.75 PPS
- Foul on One-And-One (OAO): (1 FT)(75%)(1-75%) + (2 FT)(75%)(75%) = 1.3125 PPS*
- Foul on Double Bonus (DB): (2 FT)(75%) = 1.50 PPS

*One-And-One would equate to a 5.25 system score, but will round down to nearest whole number (5) so the desired expecation is actually 1.25 (slighlty different than the true mathematical value)

In [101]:
shots_q = f"""
    SELECT Home_Team as Team, Home_Type as Type, Home_Points as Points FROM data
    WHERE Home_Type in ('T1','T2','F2','F2A','F3','F3A','OAO','DB')
    UNION ALL
    SELECT Away_Team as Team, Away_Type as Type, Away_Points as Points FROM data
    WHERE Away_Type in ('T1','T2','F2','F2A','F3','F3A','OAO','DB')
"""
con.sql(shots_q)

stats_q = f"""
    WITH shots AS ({shots_q}),
    Math AS (
        SELECT * FROM (
            VALUES
                ('T1', 3), ('T2', 6), ('F2', 6), ('F2A', 11), ('F3', 9), ('F3A', 15), ('OAO', 5), ('DB', 6)
        ) AS t(Type, Math_System_Score)
    ),
    Agg AS (
        SELECT s.Type,
               COUNT(*) AS Shots,
               AVG(Points) AS PPS,
               STDDEV_SAMP(Points) AS SD,
               m.Math_System_Score / 4.0 AS Expected_PPS
        FROM shots s
        JOIN Math m ON s.Type = m.Type
        GROUP BY s.Type, m.Math_System_Score
    ),
    T AS (
        SELECT *, (PPS - Expected_PPS) / (SD / SQRT(Shots)) AS t_stat, (Shots - 1) AS df
        FROM Agg
    )
    SELECT
        Type,
        Shots,
        printf('%.2f', PPS)          AS PPS,
        printf('%.2f', Expected_PPS) AS Expected_PPS,
        printf('%.0f', 4*Expected_PPS) AS New_System_Score,
        printf('%.2f', SD) AS SD,
        printf('%.2f', t_stat)       AS t_st,
        ROUND(t_pvalue(t_stat, df), 4) AS p_value,
        t_pvalue(t_stat, df) < 0.05/(SELECT COUNT(*) FROM Agg) AS significance
    FROM T
    ORDER BY CASE Type
        WHEN 'F2'  THEN 1
        WHEN 'F2A' THEN 2
        WHEN 'F3'  THEN 3
        WHEN 'F3A' THEN 4
        WHEN 'OAO' THEN 5
        WHEN 'DB'  THEN 6
        WHEN 'T1'  THEN 7
        WHEN 'T2'  THEN 8
    END
"""

# Persist as a view so it acts like a queryable table/DB object
con.sql(stats_q)

┌─────────┬───────┬─────────┬──────────────┬──────────────────┬─────────┬─────────┬─────────┬──────────────┐
│  Type   │ Shots │   PPS   │ Expected_PPS │ New_System_Score │   SD    │  t_st   │ p_value │ significance │
│ varchar │ int64 │ varchar │   varchar    │     varchar      │ varchar │ varchar │ double  │   boolean    │
├─────────┼───────┼─────────┼──────────────┼──────────────────┼─────────┼─────────┼─────────┼──────────────┤
│ F2      │  1847 │ 1.46    │ 1.50         │ 6                │ 0.64    │ -2.49   │  0.0129 │ false        │
│ F2A     │   456 │ 2.71    │ 2.75         │ 11               │ 0.45    │ -1.86   │   0.064 │ false        │
│ F3      │    77 │ 2.27    │ 2.25         │ 9                │ 0.84    │ 0.24    │  0.8124 │ false        │
│ F3A     │    23 │ 3.91    │ 3.75         │ 15               │ 0.29    │ 2.71    │  0.0127 │ false        │
│ OAO     │   412 │ 1.26    │ 1.25         │ 5                │ 0.87    │ 0.34    │  0.7355 │ false        │
│ DB      │   326 │

Looking at the Foul Study via Type Breakdown, there are some telling findings.
1. There is an alternative way to assess 7s/11s. Upon initial glance, the 8 different foul buckets have varying Expected PPS and therefore do not necessarily belong to a strict 7 or 11 bucket.
2. Using Bonferroni p-values (0.05/8 groups) for a significance level, none of the p-values for actual PPS values found are less than the alpha-level of 0.00625 (when looking at a one-sample t-test with known mean of the Mathetmatical Expected PPS). This indicates that there is no statistical significance of the Actual PPS differing from the Mathematical Expected PPS and can presumably carry on with the New_System_Score values for the new buckets when these shots occur.
3. Some awarenes regarding p-values... F2, F2A, F3A, and DB have p-vales <0.10 which could raise some eyebrows. BUT! These shots need to fall into a bucket of some sort anyway so if choosing the next adjacent 0.25 PPS actual/1 PPS system value (ex: F2 being considered in the 1.25/5 lower bound bucket and the 1.75/7 bucket), these significances would not pass the alpha established via bonferroni.
4. Admittedly, F3, F3A, T1, and T2 do not have a high sample count (Shots) and so should be considered with some caution. Since these values are trending toward the mathematical expectation, moving forward with this is done with confidence in spite of this caution. So while T1, F3 and F3A could fall into an upper bound bucket (1.00, 2.50, 4.00, respectively) since they do not fail this upper bound's t-test, encourages me not to use that number quite yet.
5. It is worth noting some intuitional curiosity, that the F3 and F3A could fit into the upper bound since *generally* players that shoot 3PA are better shooters and therefore better FT shooters which could mean a greater value is to be expected. But for now, sticking with the buckets chosen previously.

As seen in the table above, all 8 categories of Foul Type's System Score reflects what the Mathematical System Score should be. This indicates a new way to track foul occurrences in games!

However, since the "process" ends once the foul is called, it is preferred if 2PA and 3PA are merged into a single bucket. The mathematical expecation for this is the weighted average of their previous individual mathematical expectations weighted by the rate at which an and-1 does/does not occur for each.

In [102]:
shooting = f"""
    WITH totals AS ({shots_q}),
    Math AS (
        SELECT * FROM (
            VALUES
                ('DB', 6), ('OAO', 5), ('T1', 3), ('T2', 6), ('F_on_2PA', 7), ('F_on_3PA', 11)
        ) AS t(Type, Math_System_Score)
    ),
    typed AS (
        SELECT
            CASE WHEN Type IN ('F2', 'F2A') THEN 'F_on_2PA'
                WHEN Type IN ('F3', 'F3A') THEN 'F_on_3PA'
            ELSE Type
            END as Shooting_Type,
            Points
        FROM totals t
    ),
    combined AS (
        SELECT
            t.Shooting_Type,
            sum(t.Points) as Points,
            count(*) as Shots,
            ROUND(AVG(t.Points), 2) as PPS,
            m.Math_System_Score/4 as Expected_PPS,
            m.Math_System_Score as New_System_Score,
            STDDEV_SAMP(t.Points) as SD
        FROM typed t
        JOIN Math m on t.Shooting_Type = m.Type
        GROUP BY t.Shooting_Type, m.Math_System_Score
    ),
    stats AS (
        SELECT *, (PPS - Expected_PPS) / (SD / SQRT(Shots)) AS t_stat, (Shots - 1) AS df
        FROM combined
    )
    SELECT
        Shooting_Type,
        Shots,
        printf('%.2f', PPS)          AS PPS,
        printf('%.2f', Expected_PPS) AS Expected_PPS,
        printf('%.2f', SD) AS SD,
        printf('%.2f', t_stat)       AS t_st,
        ROUND(t_pvalue(t_stat, df), 4) AS p_value,
        t_pvalue(t_stat, df) < 0.05/(SELECT COUNT(*) FROM combined) AS significance
    FROM stats
    ORDER BY CASE Shooting_Type
        WHEN 'F_on_2PA'  THEN 1
        WHEN 'F_on_3PA' THEN 2
        WHEN 'OAO'  THEN 3
        WHEN 'DB' THEN 4
        WHEN 'T1' THEN 5
        WHEN 'T2'  THEN 6
    END
"""

con.sql(shooting)

┌───────────────┬───────┬─────────┬──────────────┬─────────┬─────────┬─────────┬──────────────┐
│ Shooting_Type │ Shots │   PPS   │ Expected_PPS │   SD    │  t_st   │ p_value │ significance │
│    varchar    │ int64 │ varchar │   varchar    │ varchar │ varchar │ double  │   boolean    │
├───────────────┼───────┼─────────┼──────────────┼─────────┼─────────┼─────────┼──────────────┤
│ F_on_2PA      │  2303 │ 1.71    │ 1.75         │ 0.79    │ -2.44   │  0.0146 │ false        │
│ F_on_3PA      │   100 │ 2.65    │ 2.75         │ 1.02    │ -0.98   │  0.3287 │ false        │
│ OAO           │   412 │ 1.26    │ 1.25         │ 0.87    │ 0.23    │  0.8165 │ false        │
│ DB            │   326 │ 1.44    │ 1.50         │ 0.63    │ -1.73   │  0.0854 │ false        │
│ T1            │     3 │ 0.67    │ 0.75         │ 0.58    │ -0.24   │  0.8327 │ false        │
│ T2            │    72 │ 1.53    │ 1.50         │ 0.58    │ 0.44    │  0.6625 │ false        │
└───────────────┴───────┴─────────┴─────

The alpha value using Bonferroni p-values is now (0.05/6 groups) instead. Some notes
1. While greater, it still holds that there is no significant difference for the previously used OAO, DB, T1, and T2. 
2. F_on_2PA and F_on_3PA both show no significant difference and could presumably be used for their pre-existing 7 and 11 bucket values
3. F_on_3PA did not reject the null when considering a lower bound which could be due to a still existing low sample size (100) and the much larger standard deviation due to a higher standard deviation.
4. Regardless, having a slightly higher Expected PPS/System Score value is more likely a better altnerative if deciding to combine due to the assumed negative conotation around fouling a 3-point shooter (punish the defense!).